# Milestone 1: universe and events

This notebook imports `earnings_vol` and reads the committed tables in `data/derived/`. It defines no logic of its own. The walkthrough is `docs/milestones/01-universe-and-events.md`.

To rebuild the tables from the sources (it needs the Dolt clones and the network):

```
uv run python -m earnings_vol.ingest.run
uv run python -m earnings_vol.events.run
uv run python -m earnings_vol.measures.run
uv run python tools/export_app_data.py
```

Contract schema version: `0.3.0`.

## 1. Day 0 on the NYSE clock (ADR 0002 §2)
Apple reported after the close on Thursday 2025-10-30, so day -1 is 10-30 and day 0 is Friday 10-31. Alphabet's calendar row of 2024-10-29 said BMO; the press release says after the close, which moves day 0 one session later.

In [ ]:
from datetime import date

from earnings_vol.events import calendar

eras = {"weekly_from": "2019-02-09", "mwf_from": "2020-01-06",
        "daily_from": "2024-09-11", "data_end": "2026-10-26"}
table = calendar.build_session_table(date(2024, 1, 1), date(2025, 12, 31), eras)
sessions = calendar.Sessions(table["session_date"].tolist())
for d, label in [(date(2025, 10, 30), "AMC"), (date(2024, 10, 29), "BMO"),
                 (date(2024, 10, 29), "AMC"), (date(2025, 5, 3), None)]:
    day0, day_m1 = calendar.day0(d, label, sessions)
    print(d, label, "-> day -1", day_m1, " day 0", day0)


## 2. The EPS-period match (ADR 0002 §3)
Costco's quarters end mid-month and are stored as month-ends, so its May report (2025-05-29) comes 2 days *before* the stored end 2025-05-31. A period's window therefore opens early in its month: 14 days after the first, the rule in `params.yaml` (the 14 days keep a slow reporter's report out of the next period's window until it is 74 days late). The March report is missing from the calendar: the February period could reach 05-29 (lag 90), but the May period is nearer, so periods and events are paired one-to-one, nearest in total, and February is the orphan. It does not take the next quarter's report.

In [ ]:
from earnings_vol import params
from earnings_vol.events.eps import match_periods

em = params.load()["eps_match"]   # window_start "month_start+14", lag_max_days 100
out = match_periods([date(2024, 11, 30), date(2025, 2, 28), date(2025, 5, 31), date(2025, 8, 31)],
                    [(date(2024, 12, 12), "Q1"), (date(2025, 5, 29), "Q3"), (date(2025, 9, 25), "Q4")],
                    window_start=em["window_start"], lag_max=em["lag_max_days"],
                    calendar_start=params.load().day("eps_match", "calendar_start"))
for period, (event, status) in sorted(out.items()):
    print(period, event, status)


## 3. The committed tables and the sample funnel
The same function that writes `app/public/data/sample_funnel.json`. Read the counts with its caveats, printed last. **Survivorship:** the 50 companies are today's largest S&P 500 members, so firms that shrank or left the index since 2020 are missing and firms that grew are over-represented.

In [ ]:
from earnings_vol import params, schema, store
from earnings_vol.export import funnel

derived = params.ROOT / "data" / "derived"
t = {n: store.read_table(derived / f"{n}.parquet", store.committed_schema(schema.REGISTRY[n]))
     for n in ("events", "eps", "session", "iv_daily", "event_iv", "timing_check")}
f = funnel.funnel_data(t["events"], t["iv_daily"], t["eps"], t["session"], t["event_iv"],
                       t["timing_check"], params.load(), provenance={})
for s in f["steps"]:
    print(f"{s['n_events']:5d}  {s['label']}")
print(f["timing_status_counts"])
for c in f["caveats"]:
    print("Caveat:", c)
